### Toy Transformer - Sequence Reversal

In [5]:
import math
import torch
import torch.nn as nn
import torch.optim as optim

In [10]:
# Positional encoding of input sequence
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=100):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(1))

    def forward(self, x):
        # x: (seq_len, batch, d_model)
        return x + self.pe[:x.size(0)]
    
# Generate random input and target sequences for training
def generate_batch(batch_size, seq_len, vocab_size):
    x = torch.randint(1, vocab_size, (batch_size, seq_len))
    y = torch.flip(x, dims=[1])
    return x, y

In [11]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0
        
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        
        # Learned projection matrices
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        
        self.W_o = nn.Linear(d_model, d_model)

    def forward(self, query, key, value, mask=None):
        # query,key,value shape: (seq_len, batch, d_model)

        seq_len, batch_size, _ = query.size()
        
        # 1) Linear projections
        Q = self.W_q(query)
        K = self.W_k(key)
        V = self.W_v(value)

        # 2) Split into heads
        # -> (batch, num_heads, seq_len, d_k)
        Q = Q.transpose(0,1).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1,2)
        K = K.transpose(0,1).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1,2)
        V = V.transpose(0,1).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1,2)

        # 3) Scaled dot-product attention
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        # scores: (batch, heads, seq_len, seq_len)

        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))

        attn = torch.softmax(scores, dim=-1)
        context = torch.matmul(attn, V)
        # context: (batch, heads, seq_len, d_k)

        # 4) Concatenate heads
        context = context.transpose(1,2).contiguous().view(batch_size, seq_len, self.d_model)

        # 5) Final linear projection
        output = self.W_o(context)

        # Return to (seq_len, batch, d_model)
        return output.transpose(0,1)

In [12]:
class SublayerConnection(nn.Module):
    """
    Residual connection followed by layer normalization.
    """
    def __init__(self, d_model, dropout=0.1):
        super().__init__()
        self.norm = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, sublayer):
        # Apply sublayer, dropout, residual, then norm
        return self.norm(x + self.dropout(sublayer(x)))
    
class PositionwiseFeedForward(nn.Module):
    def __init__(self, d_model, d_ff=512, dropout=0.1):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.linear2(self.dropout(torch.relu(self.linear1(x))))
    
class EncoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff=512, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.feed_forward = PositionwiseFeedForward(d_model, d_ff, dropout)

        self.sublayer1 = SublayerConnection(d_model, dropout)
        self.sublayer2 = SublayerConnection(d_model, dropout)

    def forward(self, x, mask=None):
        # Self-attention sublayer
        x = self.sublayer1(x, lambda x: self.self_attn(x, x, x, mask))
        # Feed-forward sublayer
        x = self.sublayer2(x, self.feed_forward)
        return x
    
class DecoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff=512, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.cross_attn = MultiHeadAttention(d_model, num_heads)
        self.feed_forward = PositionwiseFeedForward(d_model, d_ff, dropout)

        self.sublayer1 = SublayerConnection(d_model, dropout)
        self.sublayer2 = SublayerConnection(d_model, dropout)
        self.sublayer3 = SublayerConnection(d_model, dropout)

    def forward(self, x, memory, src_mask=None, tgt_mask=None):
        # Masked self-attention (decoder)
        x = self.sublayer1(x, lambda x: self.self_attn(x, x, x, tgt_mask))
        # Encoder-decoder cross attention
        x = self.sublayer2(x, lambda x: self.cross_attn(x, memory, memory, src_mask))
        # Feed-forward
        x = self.sublayer3(x, self.feed_forward)
        return x
    
class Encoder(nn.Module):
    def __init__(self, layer, N):
        super().__init__()
        self.layers = nn.ModuleList([layer for _ in range(N)])
        self.norm = nn.LayerNorm(layer.self_attn.d_model)

    def forward(self, x, mask=None):
        for layer in self.layers:
            x = layer(x, mask)
        return self.norm(x)

class Decoder(nn.Module):
    def __init__(self, layer, N):
        super().__init__()
        self.layers = nn.ModuleList([layer for _ in range(N)])
        self.norm = nn.LayerNorm(layer.self_attn.d_model)

    def forward(self, x, memory, src_mask=None, tgt_mask=None):
        for layer in self.layers:
            x = layer(x, memory, src_mask, tgt_mask)
        return self.norm(x)
    
class Transformer(nn.Module):
    def __init__(self, vocab_size, d_model=128, num_heads=4, num_layers=2, d_ff=512):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoder = PositionalEncoding(d_model)
        self.pos_decoder = PositionalEncoding(d_model)

        encoder_layer = EncoderLayer(d_model, num_heads, d_ff)
        decoder_layer = DecoderLayer(d_model, num_heads, d_ff)

        self.encoder = Encoder(encoder_layer, num_layers)
        self.decoder = Decoder(decoder_layer, num_layers)

        self.output_proj = nn.Linear(d_model, vocab_size)

    def forward(self, src, tgt, src_mask=None, tgt_mask=None):
        # src,tgt: (seq_len, batch)
        src = self.pos_encoder(self.embedding(src))
        tgt = self.pos_decoder(self.embedding(tgt))

        memory = self.encoder(src, src_mask)
        out = self.decoder(tgt, memory, src_mask, tgt_mask)

        return self.output_proj(out)
    
def subsequent_mask(size):
    mask = torch.triu(torch.ones(size, size), diagonal=1).bool()
    return ~mask  # True where allowed

In [13]:
model = Transformer(vocab_size=20)

src = torch.randint(1, 20, (8, 32))  # (seq_len, batch)
tgt = torch.randint(1, 20, (8, 32))

tgt_mask = subsequent_mask(8)

out = model(src, tgt, tgt_mask=tgt_mask)

print(out.shape)
# torch.Size([8, 32, 20])

torch.Size([8, 32, 20])


In [16]:
device = "cuda" if torch.cuda.is_available() else "cpu"

vocab_size = 20
seq_len = 8
model = Transformer(vocab_size).to(device)

optimizer = optim.Adam(model.parameters(), lr=3e-4)
criterion = nn.CrossEntropyLoss()

for step in range(1000):
    src, tgt = generate_batch(batch_size=32, seq_len=seq_len, vocab_size=vocab_size)
    src, tgt = src.to(device), tgt.to(device)

    # decoder input = target shifted right
    tgt_input = torch.zeros_like(tgt)
    tgt_input[:,1:] = tgt[:,:-1]

    logits = model(src, tgt_input)
    loss = criterion(logits.view(-1, vocab_size), tgt.transpose(0,1).reshape(-1))

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if step % 100 == 0:
        print("step", step, "loss", loss.item())

step 0 loss 3.1773314476013184
step 100 loss 2.9514944553375244
step 200 loss 2.971388339996338
step 300 loss 2.9626972675323486
step 400 loss 2.9562275409698486
step 500 loss 2.941563606262207
step 600 loss 2.9604737758636475
step 700 loss 2.955861806869507
step 800 loss 2.957118034362793
step 900 loss 2.936281442642212


In [17]:
model.eval()
src, tgt = generate_batch(1, seq_len, vocab_size)
src = src.to(device)

tgt_input = torch.zeros((1, seq_len), dtype=torch.long).to(device)

for i in range(seq_len):
    logits = model(src, tgt_input)
    next_token = logits[i,0].argmax().item()
    tgt_input[0,i] = next_token

print("Input: ", src.cpu().numpy())
print("Predicted:", tgt_input.cpu().numpy())
print("Target:   ", torch.flip(src, dims=[1]).cpu().numpy())

IndexError: index 1 is out of bounds for dimension 0 with size 1